In [1]:
import pandas as pd
import numpy as np

In [4]:
# --- 1. Production Yields par métal ---
production_yields = {
    "Cobalt": 0.439346565686978,
    "Copper": 0.834745762711864,
    "Graphite": 1,
    "Lithium": 0.699170470915539,
    "Neodymium": 0.598543913989968,
    "Nickel": 0.787554996857322
}

# --- 2. Charger les données ---
file_path = 'data/Scenario_calculation_220526.xlsx'
df_demand = pd.read_excel(file_path, 'DEMAND')
df_production = pd.read_excel(file_path, 'PRODUCTION')

# Filtrer DEMAND pour le Canada uniquement
df_demand_can = df_demand[df_demand['Country'] == 'Canada'].copy()

# Créer des index pour des lookups optimisés
demand_index = df_demand_can.set_index(['Year', 'Metal', 'Demand scenario CER'])
production_index = df_production.set_index(['Metal', 'Year', 'Production scenario CCI', 'Production scenario CCI|Refining CAN share'])

# --- 3. Paramètres ---
years = range(2025, 2041)
metals = df_demand_can['Metal'].dropna().unique()
demand_scenarios = df_demand_can['Demand scenario CER'].dropna().unique()

# Dictionnaire pour mapper Supply scenario aux combinaisons VALIDES
supply_to_valid_combinations = {
    "NIMBY": [("Existing production", "Current shares")],
    "Status Quo": [("Existing production", "Current shares")],
    "Expansion Extractive": [("Production potential", "Current shares")],
    "Expansion Extractive+Circularity": [("Production potential", "Current shares")],
    "Integrated": [("Production potential", "100% CAN")]
}

results = []

# --- 4. Calculs ---
for year in years:
    for metal in metals:
        for demand_scenario in demand_scenarios:
            # Récupérer Demand_value_t (métal raffiné)
            try:
                demand_value = demand_index.loc[(year, metal, demand_scenario), 'Demand_value_t']
            except (KeyError, TypeError):
                demand_value = np.nan

            # Récupérer le production yield pour ce métal
            yield_factor = production_yields.get(metal, 1.0)  # 1.0 par défaut si métal inconnu

            for supply_scenario in supply_to_valid_combinations:
                for (prod_scenario, refining_share) in supply_to_valid_combinations[supply_scenario]:
                    # --- Cas NIMBY : tout en RoW ---
                    if supply_scenario == "NIMBY":
                        mining_CAN = 0
                        refining_CAN = 0
                        # Mining RoW = (Demand / yield) car tout le minerai vient du RoW
                        mining_ROW = (demand_value / yield_factor) if (not pd.isna(demand_value) and yield_factor > 0) else np.nan
                        refining_ROW = demand_value if not pd.isna(demand_value) else np.nan

                    # --- Autres cas ---
                    else:
                        # Récupérer Production_mining_CAN_t
                        try:
                            mining_data = production_index.loc[(metal, year, prod_scenario)]
                            mining_CAN = mining_data['Production_mining_CAN_t'].iloc[0] if len(mining_data) > 0 else np.nan
                        except (KeyError, TypeError):
                            mining_CAN = np.nan

                        # Récupérer Production_refining_CAN_t
                        try:
                            refining_data = production_index.loc[(metal, year, prod_scenario, refining_share)]
                            refining_CAN = refining_data['Production_refining_CAN_t']
                        except (KeyError, TypeError):
                            refining_CAN = np.nan

                        # Calculer RoW
                        # Refining : RoW = Demand - Refining_CAN (1:1)
                        refining_ROW = max(0, demand_value - refining_CAN) if (not pd.isna(demand_value) and not pd.isna(refining_CAN)) else np.nan

                        # Mining : RoW = (Demand / yield) - Mining_CAN (car il faut plus de minerai pour 1kg de raffiné)
                        if not pd.isna(demand_value) and not pd.isna(mining_CAN) and yield_factor > 0:
                            mining_ROW = max(0, (demand_value / yield_factor) - mining_CAN)
                        else:
                            mining_ROW = np.nan

                    # Ajouter au résultat
                    results.append({
                        'Year': year,
                        'Metal': metal,
                        'Demand scenario CER': demand_scenario,
                        'Production scenario CCI': prod_scenario,
                        'Production scenario CCI|Refining CAN share': refining_share,
                        'Supply scenario': supply_scenario,
                        'Demand_value_t': demand_value,
                        'Production_mining_CAN_t': mining_CAN,
                        'Production_refining_CAN_t': refining_CAN,
                        'Production_mining_RoW_t': mining_ROW,
                        'Production_refining_RoW_t': refining_ROW,
                        'Production_yield': yield_factor  # Optionnel : pour référence
                    })

# --- 5. Sauvegarder les résultats ---
df_results = pd.DataFrame(results)
df_results.to_excel('calculated_scenarios_with_yields.xlsx', index=False)
print("✅ Calcul terminé. Résultats sauvegardés dans **calculated_scenarios_with_yields.xlsx**")

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3716\2590653538.py:66: PerformanceWarning: indexing past lexsort depth may impact performance.
  mining_data = production_index.loc[(metal, year, prod_scenario)]


✅ Calcul terminé. Résultats sauvegardés dans **calculated_scenarios_with_yields.xlsx**
